# 🗄️ KV-Cache and Paged Attention

Cache past keys/values so each new token isn't recomputed — and watch it balloon.

> ▶︎ In Colab: **Runtime → Run all** — this notebook runs top to bottom with no setup.

In [ ]:
%pip install -q numpy

In [ ]:
# KV-cache size = layers * 2 (K and V) * heads * head_dim * tokens * bytes.
def kv_cache_gb(layers, heads, head_dim, tokens, bytes_per=2):
    elems = layers * 2 * heads * head_dim * tokens
    return elems * bytes_per / 1e9

# A 70B-class model at growing context lengths.
for tokens in [1_000, 8_000, 32_000, 128_000]:
    gb = kv_cache_gb(layers=80, heads=64, head_dim=128, tokens=tokens)
    print(f'{tokens:>8,} tokens -> {gb:6.1f} GB of KV-cache')

## Try it

The cache can dwarf the weights. Serving many users at once multiplies it — see why memory, not FLOPs, is the bottleneck.

In [ ]:
per_user = kv_cache_gb(80, 64, 128, 32_000)
for users in [1, 10, 50]:
    print(f'{users:>3} concurrent users -> {users * per_user:6.1f} GB just for KV-cache')
print('PagedAttention exists to stop this memory from being wasted in fragments.')

## Takeaway

- The KV-cache is the real memory cost of long-context serving.
- Paging it (like an OS pages RAM) is the core of fast inference engines.

## 🚀 Your move

Build: compute the KV-cache size for a 70B-class model at 32k context (layers × 2 × heads × dim × tokens × bytes). The number explains why your long chats get expensive — and why paging matters.